In [1]:
import sys
import os
import json
import pandas as pd
import ace_lib as ace
import nest_asyncio
import asyncio
from openai import AsyncOpenAI
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
nest_asyncio.apply()
_llm_instance = None
my_api_key = None
data_fields = None
operators = None
dataset_ids = None


async def call_llm(prompt):
    global my_api_key
    """
    Interface with the LLM API to process the given prompt.
    Consultants will modify this function to use their preferred LLM.
    """
    if my_api_key is None:
        try:
            with open(project_root + '/credential.txt', 'r') as f:
                credentials = json.load(f)
            my_api_key = credentials[0]
        except (FileNotFoundError, json.JSONDecodeError, IndexError) as e:
            raise ValueError(f"Error reading credentials from credential.txt: {e}")
    try:
        # Example: OpenAI GPT (consultants can replace this with their own LLM logic)
        client = AsyncOpenAI(
            base_url="https://ark.cn-beijing.volces.com/api/v3",
            api_key=my_api_key,
            # api_key = "your-api-key"
        )
        
        # Send the prompt to the chat completion endpoint
        response = await client.chat.completions.create(
            # model="gpt-4",  # Specify the model
            # messages=[
            #     {"role": "user", "content": prompt}
            # ]
            model="doubao-seed-1-6-thinking-250715",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error calling LLM: {e}")
        return None

# Generate English Description for Alpha
async def generate_alpha_description(alpha_id, brain_session):
    global operators, data_fields, dataset_ids
    alpha_details = brain_session.get(f"https://api.worldquantbrain.com/alphas/{alpha_id}").json()
    alpha_expression = alpha_details['regular']['code']

    # If needed get operators or other data
    if operators is None:
        operators = ace.get_operators(brain_session)
    
    # dataset_ids = ['pv1', 'shortinterest3']
    
    data_fields = pd.concat(
        [ace.get_datafields(brain_session, region='ASI', universe='MINVOL1M', dataset_id=dataset_id, data_type='ALL') for dataset_id in dataset_ids],
        ignore_index=True
    )

    # Generate English description using call_llm
    prompt = f"""Describe the following alpha in plain English:\n
    Alpha: {alpha_expression}. Here you can find used operators {operators[operators['scope']=='REGULAR'].to_json()}
    and data {data_fields.to_json()}."""
    # description = await call_llm(prompt)
    # return description.strip()
    return prompt

# Generate new Alphas based on generated description
async def generate_new_alphas(alpha_description, brain_session):
    global operators, data_fields, dataset_ids
    num_alphas = 5

    # If needed get operators or other data
    if operators is None:
        print("Get operators second time")
        operators = ace.get_operators(brain_session)
    if data_fields is None:
        print("Get data fields second time")
        dataset_ids = ['pv1', 'shortinterest3',]
        data_fields = pd.concat(
            [ace.get_datafields(brain_session, region='EUR', universe='TOP2500', dataset_id=dataset_id, data_type='ALL') for dataset_id in dataset_ids],
            ignore_index=True
        )

    prompt = f"""
    Based on the following description: '{alpha_description}', generate {num_alphas} new alpha expressions using the provided operators and data.

    Operators: {operators[operators['scope']=='REGULAR'].to_json()}, data {data_fields.to_json()} where id is data field name
    Important: You can use type=MATRIX field by itself, as input to Arithmetic, 
    Cross Sectional, Time Series operators, With Logical and Transformational operators, As group in Group operators, with bucket().
    You can’t use type=VECTOR field by itself. You only can use type=VECTOR field as input to Vector operator. Then you can treat it as a MATRIX field.
    Always wrap type=VECTOR data in category=Vector operator.
    You can’t use type=GROUP field by itself. You need to use it as “group” parameter in Group operator.

    Provide only {num_alphas} alpha expressions, they should not be the same.
    """
    response = await call_llm(prompt)
    return response.strip()


async def main():
    global dataset_ids
    # Start Brain session
    brain_session = ace.start_session()

    # List your alpha IDs
    # alpha_ids = ["..."]
    alpha_ids = ["E5OaE8r9"]
    dataset_ids = ['sentiment21', 'other455']

    for alpha_id in alpha_ids:
        print(f"Processing Alpha ID: {alpha_id}")

        # Step 1: Generate English description of the alpha
        alpha_description = await generate_alpha_description(alpha_id, brain_session)
        print(f"\nAlpha Description:\n{alpha_description}")

        # Step 2: Generate new alphas based on the description
        # new_alphas = await generate_new_alphas(alpha_description, brain_session)
        
        # Logic to simulate and tag
#         simulate_data = ace.generate_alpha(brain_session, regular=...)
#         simulation_result = ace.simulate_single_alpha(brain_session, simulate_data)
#         child_alpha_id = simulation_result['alpha_id']
#         ace.set_alpha_properties(brain_session, child_alpha_id, tags = [f"alpha_id"], regular_desc = generate_alpha_description(child_alpha_id, brain_session)) 
        
        # print(f"\nNew Alphas:\n{new_alphas}")

In [2]:
asyncio.run(main())

Processing Alpha ID: E5OaE8r9

Alpha Description:
Describe the following alpha in plain English:

    Alpha: group_neutralize(-ts_product(winsorize(ts_backfill(snt21_3neg_conf_low, 120), std=4), 5),densify(oth455_competitor_n2v_p10_q200_w1_kmeans_cluster_10)). Here you can find used operators {"name":{"1":"add","4":"multiply","7":"sign","10":"subtract","13":"log","16":"max","19":"abs","22":"divide","25":"min","28":"signed_power","31":"inverse","34":"sqrt","37":"s_log_1p","40":"reverse","43":"power","46":"densify","48":"or","51":"and","54":"not","57":"is_nan","60":"less","63":"equal","66":"greater","69":"if_else","72":"not_equal","75":"less_equal","78":"greater_equal","81":"ts_corr","83":"ts_zscore","85":"ts_product","87":"ts_std_dev","89":"ts_backfill","91":"days_from_last_change","93":"last_diff_value","95":"ts_scale","97":"ts_step","99":"ts_sum","101":"inst_tvr","103":"ts_decay_exp_window","105":"ts_av_diff","107":"ts_mean","109":"ts_arg_max","111":"ts_rank","113":"ts_delay","115":"t